<a href="https://colab.research.google.com/github/PontiacGTO2006/machine-learning/blob/main/sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preparing Data
So far, you've created a simple system for making a call about whether a review has a positive or negative overall rating. This system works if the review uses words you've pre-determined as positive or negative.

But what about reviews with more complicated words or phrases? To better understand text from the reviews, you'll build a neural network trained using data from reviews. This network will create a more nuanced understanding of the review data.

There are three main stages of creating this network:


*   Importing and cleaning up the data
*   Vectorizing the data
*   Creating and training the network




In [29]:
# mount google drive to save files persistently
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

## Importing Libraries & Tools
In the code cell below, you'll import all the libraries and tools required for the sentiment analysis project.

In [1]:
# Import the pandas library.
import pandas as pd
# Import the numpy library.
import numpy as np

# Import the string library.
import string
# Import the punctuation library.
from string import punctuation
# Import the nltk library.
import nltk
# Import the stopwords.
from nltk.corpus import stopwords
# Download the stopwords
nltk.download('stopwords')

# Import the TensorFlow library.
import tensorflow as tf

# Import the Input, Dense, Dropout layers.
from tensorflow import keras # need to import keras for next line to work
from keras.layers import Input, Dense, Dropout
# Import the Sequential model.
from keras import Sequential

# Import the sklearn library.
import sklearn
# Import the train_test_split model selection.
from sklearn.model_selection import train_test_split
# Import the CountVectorizer, TfidfTransformer, TfidfVectorizer.
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer

# Import the joblib library.
import joblib

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


# Preparing the Data
To get started, you'll first import a dataset of reviews to your project. Next, you'll prepare the data to get the information required to train a network.

## Importing the Data
To import the data, you'll download the "Reviews.csv" file and load it into your project using the **.read_csv** function.

In [10]:
# Use the .read_csv function to read the "Reviews.csv" data into the notebook.
data = pd.read_csv('extracted_data/Reviews.csv')

# Use data.head() to print out the first few lines of data.
data.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


## Extracting the Data
Now that you've imported the data, you'll extract only the parts you need to train the network.

In [11]:
# Add the 'UserId', 'Id', and 'Time' to the drop function to drop them.
data = data.drop(['UserId', 'Id', 'Time'], axis=1)

# Use dropna() to drop empty rows.
data.dropna(inplace=True)

# Print the first few lines of the data.
data.head()

,ProductId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Summary,Text
0,B001E4KFG0,delmartian,1,1,5,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,B00813GRG4,dll pa,0,0,1,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,B000LQOCH0,"Natalia Corres ""Natalia Corres""",1,1,4,"""Delight"" says it all",This is a confection that has been around a fe...
3,B000UA0QIQ,Karl,3,3,2,Cough Medicine,If you are looking for the secret ingredient i...
4,B006K2ZZ7K,"Michael D. Bigham ""M. Wassir""",0,0,5,Great taffy,Great taffy at a great price. There was a wid...


## Adding a Polarity Column
Now, it's time to add a new column labeling the rating as either positive, negative, or neutral. To do this, you'll use the reviewers' star ratings. If the reviewer gave the item more than 3 stars, then it's a positive review. If the reviewer gave the item less than 3 stars, then it's a negative review. Finally, if the reviewer gave the item 3 stars, then it is neutral.

In [12]:
# Create a new column to keep track of if the review is positive negative or neutral.
data['Polarity_Rating'] = data['Score'].apply(lambda x: 'Positive' if x > 3 else('Neutral' if x==3 else 'Negative'))

# Print the first few lines of the data.
data.head()

,ProductId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Summary,Text,Polarity_Rating
0,B001E4KFG0,delmartian,1,1,5,Good Quality Dog Food,I have bought several of the Vitality canned d...,Positive
1,B00813GRG4,dll pa,0,0,1,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...,Negative
2,B000LQOCH0,"Natalia Corres ""Natalia Corres""",1,1,4,"""Delight"" says it all",This is a confection that has been around a fe...,Positive
3,B000UA0QIQ,Karl,3,3,2,Cough Medicine,If you are looking for the secret ingredient i...,Negative
4,B006K2ZZ7K,"Michael D. Bigham ""M. Wassir""",0,0,5,Great taffy,Great taffy at a great price. There was a wid...,Positive


## Sampling the Data
Now, it's time to separate the data into groups based on whether it's positive, negative, or neutral. You'll also get a sample of 8,000 reviews from each list to perform tests. 8000 is a good sample size since it won't create a data set that's too large.

In [13]:
# Separate the positive data into a group.
data_positive = data[data['Polarity_Rating'] == 'Positive']
# Separate the negative data into a group.
data_negative = data[data['Polarity_Rating'] == 'Negative']
# Separate the neutral data into a group.
data_neutral = data[data['Polarity_Rating'] == 'Neutral']

# Print out the shape of each list.
print("Negative: ", data_negative.shape)
print("Neutral: ", data_neutral.shape)
print("Positive: ", data_positive.shape)



Negative:  (82007, 8)
Neutral:  (42638, 8)
Positive:  (443756, 8)


In [14]:
# Get a sample from the positive reviews.
data_positive = data_positive.sample(8000)
# Get a sample from the negative reviews.
data_negative = data_negative.sample(8000)

# Get a sample from the neutral reviews.
data_neutral = data_neutral.sample(8000)

# Print out the shape of the new lists.
print("Negative: ", data_negative.shape)
print("Neutral: ", data_neutral.shape)
print("Positive: ", data_positive.shape)



Negative:  (8000, 8)
Neutral:  (8000, 8)
Positive:  (8000, 8)


In [15]:
# Combine the positive, negative, and neutral data lists together to create one large dataset.
data = pd.concat([data_positive, data_negative, data_neutral])

# Print the data's shape.
print(data.shape)

(24000, 8)


# Cleaning the Text
Now that you have one large list of the data, it's time to clean the actual text. You'll create a function that removes punctuation and stopwords from the text.

You'll first remove the stopwords. The string library has a built-in list of stopwords. Stopwords are words that don't contain important information and are common in English. Examples include "is," "our," "the," etc.

Next, you'll remove the punctuation from the text by creating a for loop and adding the text without the punctuation into a list called no_punctuation.

Finally, you'll return the filtered text. To do this, you'll create a for loop that splits the text into a list of filtered words and checks if the lowercase word is in the stopwords. This will ensure that the data is filtered properly.

Once your data is cleaned and prepared, you'll view the new data by creating a new column and recreating the data to work with the information you need.

In [16]:
# Create a function called text_cleanup that returns the text without the stopwords and punctuation.
def text_cleanup(text):
    get_stopwords = stopwords.words('english')
    no_punctuation = []
    for i in text:
        if i not in string.punctuation:
            no_punctuation.append(i)
    no_punctuation = ''.join(no_punctuation)

    filtered_words = []
    for word in no_punctuation.split():
        if word.lower() not in get_stopwords:
            filtered_words.append(word)
    return ''.join(filtered_words)



In [17]:
# Create a new column called reviews that cleans up the text.
data['reviews'] = data['Text'].apply(text_cleanup)

# Print the first few lines of the data.
data.head()

,ProductId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Summary,Text,Polarity_Rating,reviews
323230,B000OQ2DL4,L. Snyder,52,66,5,"Excellent to relieve muscle, joint pain, sleep...",My 80 year old Mom suffered from excruciating ...,Positive,80yearoldMomsufferedexcruciatingbackmusclepain...
195881,B006N3I29E,3boysbaseball,0,0,5,Smooth coffee,My favorite blend of coffee for my Keurig. I'v...,Positive,favoriteblendcoffeeKeurigIvetried15differentfl...
51709,B0002NYO20,txmarleygirl,0,0,5,Awesome,Had this sent to my best friend's house for ma...,Positive,sentbestfriendshousemakinglavenderfoodsreceive...
542629,B006J0WNX2,Kevin W Lape,4,4,5,This is my favorite one,"I work night shift, so an energy boost is a ne...",Positive,worknightshiftenergyboostnecessitygetlasthours...
307228,B002GWMGV6,Tracie Toy,0,0,5,awsome coffee,I agree with the other reviewer. This is the b...,Positive,agreereviewerbestcoffeedrinking2yearscantstart...


In [18]:
# Add the review data and polarity rating columns to recreate the dataset with just the needed data.
data = data[['reviews', 'Polarity_Rating']]

# Print the first few lines of the data.
data.head()

,reviews,Polarity_Rating
323230,80yearoldMomsufferedexcruciatingbackmusclepain...,Positive
195881,favoriteblendcoffeeKeurigIvetried15differentfl...,Positive
51709,sentbestfriendshousemakinglavenderfoodsreceive...,Positive
542629,worknightshiftenergyboostnecessitygetlasthours...,Positive
307228,agreereviewerbestcoffeedrinking2yearscantstart...,Positive


## One Hot Encoding    
One-hot encoding is a process of creating data categories that the network can understand and that won't have an inherent bias. In this case, you'll make a matrix of three columns (positive, negative, and neutral) and put either a 1 or 0 in the row for the review text. This way, the network can be trained using the matrix to validate it's guesses.

In [19]:
# Apply one-hot encoding to your data by using the pandas get_dummies function.
one_hot = pd.get_dummies(data["Polarity_Rating"])

# Print the first few lines of the one-hot data.
one_hot.head()

,Negative,Neutral,Positive
323230,False,False,True
195881,False,False,True
51709,False,False,True
542629,False,False,True
307228,False,False,True


In [20]:
# Combine your data and the one_hot data to combine the data into one big dataset.
data = pd.concat([data, one_hot], axis=1)

# Print the first few lines of the data.
data.head()

,reviews,Polarity_Rating,Negative,Neutral,Positive
323230,80yearoldMomsufferedexcruciatingbackmusclepain...,Positive,False,False,True
195881,favoriteblendcoffeeKeurigIvetried15differentfl...,Positive,False,False,True
51709,sentbestfriendshousemakinglavenderfoodsreceive...,Positive,False,False,True
542629,worknightshiftenergyboostnecessitygetlasthours...,Positive,False,False,True
307228,agreereviewerbestcoffeedrinking2yearscantstart...,Positive,False,False,True


In [21]:
# Drop the 'Polarity_Rating' column.
data.drop(['Polarity_Rating'], axis=1, inplace=True)

# Print the first few lines of the data.
data.head()

,reviews,Negative,Neutral,Positive
323230,80yearoldMomsufferedexcruciatingbackmusclepain...,False,False,True
195881,favoriteblendcoffeeKeurigIvetried15differentfl...,False,False,True
51709,sentbestfriendshousemakinglavenderfoodsreceive...,False,False,True
542629,worknightshiftenergyboostnecessitygetlasthours...,False,False,True
307228,agreereviewerbestcoffeedrinking2yearscantstart...,False,False,True


## Train and Test Split
Now that you have all the data cleaned up and formatted for the network, you'll split it into train and test sets. To split the data into these groups, you'll use the **sklearn** library. You'll use the built-in **train_test_split()** function to get a train and test data sets.

To feed data into the network, you need an input set and an output set. The input will be the reviews, and the output will be the polarity.

In [22]:
# Set x_rev equal to the reviews column by adding .values to the end to get the data itself out of the column.
x_rev = data["reviews"].values

# Set y_pol to the data with the reviews column dropped.
y_pol = data.drop("reviews", axis=1)

# Create your train and test datasets.
x_rev_train, x_rev_test, y_pol_train, y_pol_test = train_test_split(x_rev, y_pol, test_size=0.30, shuffle=True)

# Vectorizing
So far, you've cleaned up all the text data from the reviews and separated them into input and output sets.

Now that you have only the important words of the reviews data and a matrix to represent their polarity, you'll turn the text into numbers so that the network can understand the format.

You'll do this using a process called **Vectorizing**.

**Vectorizing** converts text data into numerical data so that a neural network can perform calculations with it.

To do this, you're going to use **sklearn**. This library has a lot of built-in libraries for reworking data into its numerical form.

This process has two phases: **fit** and **transform**.

Luckily, the sklearn library has built-in functions to do most of the work for you during this stage. You'll use a vectorizer called **CountVectorizor** that uses the count of each word to do the vectorization process.

You'll follow the steps below to create a vocabulary from the review data and then transform the input datasets into that vocabulary.

## Fit Stage
First, you'll create a vocabulary using the **.fit()** function. You'll use all the review data from the dataset and create a set of the frequency of each word. Then, you'll export that data and save it.

In [23]:
# Create a count vectorizer object.
vect = CountVectorizer()

# Set a maximum amount of features for the vectorizer to 15,000.
vect.max_features = 15000

# Add the review data to the fit function.
vect.fit(x_rev)

# Get the vectorized vocabulary.
vocab = vect.vocabulary_

# Add a print statement to print out the vocab that has been saved to a variable.
print(vocab)

{'favoriteblendcoffeekeurigivetried15differentflavorsfarfavoritesmoothoverpoweringbitterbuyingwalmartstoppedstockingfoundamazonprice': np.int64(5855), 'agreereviewerbestcoffeedrinking2yearscantstartdaywithoutbuycoffeebeandirectcantsayanythinghighlandergrogcompaniesbest': np.int64(146), 'flavorbestbunchfarreluctantlydietallowsalmondssnacktriedhookedunfortunatelystorescarryflavorfirstcasewentquicklyeverycoworkertriedwantedrecommendanyoneustryingstickdietdeprivesuslesshealthysnacks': np.int64(6476), 'infantsonlovesfoodeasygetcasesshippedfumbleindividualjarsstorequickshippinggoodpackagingiveissues': np.int64(9476), 'goodgummibearsenjoyedlongtimegettinglargequantitylittleintimidatingfirstsmallbagsmakeeasierlimitintakewonderfullittlepiecesjoyshortsmallbagsmakecandylastlonger': np.int64(7325), 'likebakerbreakfastcookiesivepurchasedamazonmochacappuccinoespeciallytastyhealthywaystartday': np.int64(10594), 'eatsunflowerseedsalmosteverydaydavidsfavoritebrandfindinglargebagsstoresdifficultpricecre

In [24]:
# Save the vocab to your Student folder.
joblib.dump(vocab, "vocab.pkl")

['vocab.pkl']

## Transform
The transform stage applies the data you'll use to the vocabulary created by the fit stage. You'll follow the steps below to transform the data into a form the network can understand.

In [25]:
# Transform the training data.
x_rev_train_v = vect.transform(x_rev_train)

# Transform the test data.
x_rev_test_v = vect.transform(x_rev_test)

In [26]:
# Transform the training data into an array.
x_rev_train_v = x_rev_train_v.toarray()

# Transform the test data into an array.
x_rev_test_v = x_rev_test_v.toarray()

In [27]:
# Print out the shape of the training dataset.
print(x_rev_test_v.shape)

# Print the shape of the test dataset.
print(x_rev_train_v.shape)

(7200, 15000)
(16800, 15000)


# Creating the Network
Up to now, you've cleaned up and prepared the data and vectorized it. Now, it's time to create the network to understand the text. Now that you have the text as number inputs and three output categories, this is a machine-learning task. Thus, you can create a network that does classification to perform sentiment analysis.

To get started, you'll create a model and set up the layers for the neural network.

## Create the Model
You'll use a **Sequential** model for the sentiment analysis. You'll create an input layer, several calculation layers, dropout layers, and an output layer.

In [28]:
# Create a sequential model.
model = Sequential()

## Input Layer
The first layer of the network, or input layer, will take all of the text input and start to perform calculations with it. This layer needs to be large so the network can learn about the capacity of data it will be working with. The one in this example will be set to 4000 neurons, but if you want to go larger, you can!

Also, for the input layer, you'll decide on an activation algorithm. **Relu** works well for these purposes because it helps the neural network learn the complicated patterns of the dataset.

In [31]:
# Add an input layer with 6000 units and an activation of 'relu'.
model.add(Dense(units=6000, activation='relu'))

## Dropout Layers
Another important aspect of this model is the dropout layers. These help prevent the network from relying too much on specific neurons and force all the neurons to perform calculations. This is important for large networks like this one! The layer is set to dropout at a rate of 0.5, which means half of the neurons.

In [32]:
# Add a dropout layer with a rate of 0.5.
model.add(Dropout(0.5))

## Calculation Layers
Once you have the input, you'll add the calculation layers. This part of the network will do the bulk of the calculations. It's important to include dropout layers in this part of the network as well so that it continues to be versatile.

In [33]:
# Add layers to the middle of the network.
model.add(Dense(units=3000, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(units=700, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(units=350, activation='relu'))
model.add(Dropout(0.5))

## Output Layer
Finally, you'll add a layer that outputs the network's decision. The number of neurons for this layer will be the number of possible categories. In this case, there are three possible outputs: positive, negative, and neutral. The activation algorithm will be softmax. In this case, the softmax function will create a probability to determine which category each review belongs to.

In [34]:
# Add a final Dense layer to represent the output with the units set to 3 (VERY LITTLE UNITS?) and activation set to "softmax".
model.add(Dense(units=16, activation='softmax'))

## Compiling the Network
Now that you have all the layers, you'll compile the network. First, you'll create a variable to hold the optimizer and then compile the network.

The optimizer will be a general-purpose algorithm called **Adam** and the loss algorithm is **categorical cross-entropy**. Since this problem involves sorting data into categories, the metric to watch is accuracy.

In [35]:
# Add the compile function that calculates the loss and uses the optimizer parameter to set the optimization algorithm.
model.compile(loss="categorical_crossentropy", optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),metrics=["accuracy"])

## Fit Data to the Network
Next, you'll train the network using the data that you've prepared. During this stage, you'll also decide on batch size and the number of epochs. Finally, you'll set the validation data to the test datasets you created.

In [36]:
# Add the fit function and set the input data for this model, the epochs to the fit stage, and shuffle the data, so the network doesn't rely on a pattern to learn.
model.fit(
    x=x_rev_train_v,
    y=y_pol_train,
    batch_size=512,
    epochs=30,
    validation_data=(x_rev_test_v, y_pol_test)
)

KeyboardInterrupt: 

## Evaluating the Network
Once the network is done training, you'll get the scores from the model and print out the accuracy. This will give you a good idea of how the network performs on the test data. This network should provide an accuracy of about 0.70, which is good.

In [ ]:
# Calculate the scores and calculate the loss and accuracy of your model.
scores = model.evaluate(x_rev_test_v, y_pol_test, verbose=1)

# Print the test accuracy of your model.
print('Test accuracy:', scores[1]) # test accuracy was only 0.4044 at last compile on 2/8/2026 (horrible)

NameError: name 'model' is not defined

## Save the Model
You'll add the code to export the model, so this way, you won't have to retrain it to use in the future.

In [ ]:
# Export your model.
model.save('sentiment_analysis_v1.keras')


NameError: name 'model' is not defined